In [ ]:
"""This is the module to demonstarte GOFHER outputting files."""

## Import:
Import the following functions/ classes:

In [1]:
from pathlib import Path

import numpy as np

from gofher.file_helper import assure_folder_exists, read_array_file
from gofher.gofher_parameters import read_gofher_parameters_from_csv
from gofher.galaxy import Galaxy
from gofher.sparcfire import gofher_params_from_sparcfire_csv

## Paths:
Set the specific paths to the fits data and galaxy.csv

We will also create helper functions to get the full path for the fits/ galaxy.csv

In [2]:
#IMPORTANT: Make sure these folder/file names reflect test file structure:
TEST_GALAXY_DIR = "NGC2347_SDSS_psf4_background_256"
TEST_SPARCFIRE_DIR = "sparcfire_r_band_output"
TEST_FITS_DIR = "fits"
TEST_CSV_FILE_NAME = "NGC2347_r.csv"

def _get_test_csv_path() -> str:
    p = Path("..").resolve()
    return str(p.joinpath("tests", "data", TEST_GALAXY_DIR, TEST_SPARCFIRE_DIR, TEST_CSV_FILE_NAME))

def _get_test_fits_path(the_band) -> str:
    """Helper function to get path to test fits file"""
    fits_file_name = f"NGC2347_{the_band}.fits"

    p = Path("..").resolve()
    return str(p.joinpath("tests", "data", TEST_GALAXY_DIR, TEST_FITS_DIR, fits_file_name))

## Set the Run specific parameters:
* `bluer_to_redder_bands` is a list of the fits wavebands ordered bluest light first to reddest light last
* `sparcfire_bulge_disk_f` is the number in range `[0,1]` that specifies the scale of the galaxy ellipse mask to use. When `sparcfire_bulge_disk_f=0` the ellipse uses the bulge major axis, when `sparcfire_bulge_disk_f=1` the ellipse uses the disk major axis, otherwise interpolates between bulge and disk major axis

In [3]:
BLUER_TO_REDDER_BANDS = ['g','r','i','z']
SPARCFIRE_BULGE_DISK_F = 0.5

## Running Gofher:

In [4]:
# Get the GOFHER parameters from the SparcFiRe galaxy.csv:
gofher_param = gofher_params_from_sparcfire_csv(_get_test_csv_path())[0]

# Create a Galaxy and add all wavebands to it:
the_galaxy = Galaxy(gofher_param)

for band in BLUER_TO_REDDER_BANDS:
    the_galaxy.construct_galaxy_band_from_fits(band,_get_test_fits_path(band))

# Run GOFHER on the galaxy:
the_galaxy.run(bluer_to_redder_bands=BLUER_TO_REDDER_BANDS,
               sparcfire_bulge_disk_f=SPARCFIRE_BULGE_DISK_F,
               area_to_consider=None,
               fail_silently_on_missing_band=False)

## Output GOFHER data:
* Figure: A figure that visualizies GOFHER's output
* CSV: A csv containing all the gofher_parameters used and the data derived from running data
* Normalizations: For each waveband, output the numpy array representing the `normalization` and the `area_to_norm`. This can be used to analyze the raw data without having to rerun GOFHER.

In [5]:
# Create an output folder:
output_folder = Path.cwd() / "test_galaxy_output"
assure_folder_exists(output_folder)

# Save a figure visualizing GOFHER's output:
the_galaxy.plot_figure(output_folder / "NGC2347.png")

# Save a csv with GOFHER's data:
the_galaxy.output_to_csv(output_folder / "NGC2347.csv")

# Save the normalization:
the_galaxy.save_normalizations(output_folder)

## To see the output go to `output_folder`

## How to read in GOFHER csv and normalization to analze data without rerunning GOFHER:

In [6]:
# Read in the gofher_parameters from the csv:
gp = read_gofher_parameters_from_csv(output_folder / "NGC2347.csv")

# Read in the normalization numpy files:
area_to_norm = read_array_file(output_folder / "area_to_norm.npy")
g_norm = read_array_file(output_folder / "g_normalization.npy")
r_norm = read_array_file(output_folder / "r_normalization.npy")

# Create the bisection masks from the gofher parameters that were read in:
pos, neg = gp.create_bisection_masks()

# Recreate the g-r diff image using the normalizations
recreated_diff = np.zeros(area_to_norm.shape)
recreated_diff[area_to_norm] = g_norm[area_to_norm] - r_norm[area_to_norm]

# Get the value on the pos and neg side:
pos_values = recreated_diff[np.logical_and(area_to_norm, pos)]
neg_values = recreated_diff[np.logical_and(area_to_norm, neg)]

# Analyze the data however you want:
print(f"pos side: mean {np.mean(pos_values)} std {np.std(pos_values)}")
print(f"neg side: mean {np.mean(neg_values)} std {np.std(neg_values)}")

pos side: mean 0.007028242287138587 std 0.013166631922623287
neg side: mean 0.00016580744777092965 std 0.016576339764720522
